
# PCA — 라그랑주 + 편미분으로 유도하는 주성분 분석

> 본 노트북의 목적은 **주성분 분석(Principal Component Analysis, PCA)** 의 추정식을  
> "분산 최대화 + 단위벡터 제약 + 라그랑주 + 편미분" 이라는 다섯 단어로 손으로 유도하고,  
> 그 결과가 곧 **공분산행렬의 고유값-고유벡터 문제** 임을 NumPy 로 직접 확인하는 것이다.

기계학습 강의노트 · 04 · 부록 B.  
**선행자료**: `OLS_notebook.ipynb` (편미분으로 정규방정식을 얻는 사고방식).



## 1.  무엇을 최대화하는가

자료 행렬 $\boldsymbol{X} \in \mathbb{R}^{n \times p}$ 가 평균 0 으로 중심화되어 있다고 하자. 단위벡터 $\boldsymbol{v} \in \mathbb{R}^{p}$ 방향으로 자료를 사영(projection)한 결과 $\boldsymbol{X v}$ 의 표본 분산은

$$ \mathrm{Var}(\boldsymbol{X v}) \;=\; \frac{1}{n-1}\,\boldsymbol{v}^{\mathsf T} \boldsymbol{X}^{\mathsf T} \boldsymbol{X v} \;=\; \boldsymbol{v}^{\mathsf T} \Sigma \boldsymbol{v} $$

이다. 여기서 $\Sigma = \boldsymbol{X}^{\mathsf T}\boldsymbol{X} / (n-1)$ 은 자료의 표본 공분산행렬이다.

PCA 의 제 1 주성분은 \"자료를 가장 길게 펼치는 방향\" — 즉 $\boldsymbol{v}^{\mathsf T}\Sigma\boldsymbol{v}$ 를 최대로 만드는 단위벡터 $\boldsymbol{v}$ 이다.

$$
\boxed{\;
\max_{\boldsymbol{v}} \;\; \boldsymbol{v}^{\mathsf T} \Sigma \boldsymbol{v}
\quad \text{subject to} \quad \boldsymbol{v}^{\mathsf T}\boldsymbol{v} = 1.
\;}
$$



## 2.  라그랑주 + 편미분

OLS 는 제약 없는 미분이었지만, PCA 는 \"$\|\boldsymbol{v}\| = 1$\" 이라는 등식 제약이 붙는다. 그래서 **라그랑주 승수법** 을 동원한다. 라그랑주 함수

$$ L(\boldsymbol{v}, \lambda) = \boldsymbol{v}^{\mathsf T}\Sigma\boldsymbol{v} - \lambda(\boldsymbol{v}^{\mathsf T}\boldsymbol{v} - 1) $$

를 $\boldsymbol{v}$ 에 대해 편미분하여 0 으로 놓으면

$$ \frac{\partial L}{\partial \boldsymbol{v}} = 2\Sigma\boldsymbol{v} - 2\lambda\boldsymbol{v} = \boldsymbol{0} $$

이고, 정리하면

$$ \boxed{\;\Sigma\boldsymbol{v} = \lambda\boldsymbol{v}\;} $$

이다. 이것은 정확히 공분산행렬 $\Sigma$ 의 **고유값-고유벡터 방정식** 이다.

두 가지 결론이 자동으로 따라온다.

1. **$\boldsymbol{v}$ 는 $\Sigma$ 의 고유벡터** — 주성분 방향이다.
2. **$\lambda$ 가 곧 분산값이다**: $\boldsymbol{v}^{\mathsf T}\Sigma\boldsymbol{v} = \lambda \boldsymbol{v}^{\mathsf T}\boldsymbol{v} = \lambda$.

따라서 \"가장 큰 분산\" 을 주는 방향은 \"가장 큰 고유값에 대응하는 고유벡터\" 이다.

> **OLS 와의 대비.**
> OLS:    $\partial S / \partial \boldsymbol{\beta} = \boldsymbol{0}$ ⟶ 정규방정식 ⟶ $\hat{\boldsymbol{\beta}} = (X^{\mathsf T}X)^{-1}X^{\mathsf T}y$.
> PCA:    $\partial L / \partial \boldsymbol{v}  = \boldsymbol{0}$ ⟶ 고유값 방정식 ⟶ $\Sigma\boldsymbol{v} = \lambda\boldsymbol{v}$.
> 두 방법 모두 \"편미분 = 0\" 한 줄에서 자동으로 풀린다.



## 3.  코드로 확인 — 2 차원 자료

상관된 2 변수 자료를 만들고, 직접 고유분해를 수행해 본다.


In [ ]:
import numpy as np
import matplotlib.pyplot as plt

rng = np.random.default_rng(seed=20260515)

# 상관된 2변수 자료
n = 300
z = rng.normal(0, 1, n)
x1 = 2.0 * z + rng.normal(0, 0.4, n)
x2 = 1.0 * z + rng.normal(0, 0.4, n)
X = np.column_stack([x1, x2])

# 평균 중심화
X_centered = X - X.mean(axis=0)
print(f"X shape: {X.shape}")
print(f"평균: {X.mean(axis=0).round(4)}")
print(f"표본 표준편차: {X.std(axis=0, ddof=1).round(4)}")



### 3.1  공분산행렬 → 고유분해

자료의 공분산행렬을 만들고, NumPy 의 `np.linalg.eig` 으로 고유값과 고유벡터를 구한다. 고유값을 큰 순서로 정렬한다.


In [ ]:
# 공분산행렬
Sigma = np.cov(X, rowvar=False, ddof=1)
print("Σ =")
print(np.round(Sigma, 4))

# 고유분해 (대칭행렬이므로 eigh 가 안전)
eigvals, eigvecs = np.linalg.eigh(Sigma)

# 큰 순서로 정렬
order = np.argsort(eigvals)[::-1]
eigvals = eigvals[order]
eigvecs = eigvecs[:, order]

print(f"\n고유값 λ (큰 순서):  {np.round(eigvals, 4)}")
print(f"\n고유벡터 (열이 v₁, v₂):")
print(np.round(eigvecs, 4))



### 3.2  sklearn 의 PCA 와 비교

손으로 푼 답과 `sklearn.decomposition.PCA` 의 결과가 일치해야 한다.
(고유벡터의 부호는 자유롭게 뒤집힐 수 있다 — 방향만 같으면 같은 답이다.)


In [ ]:
from sklearn.decomposition import PCA

pca = PCA(n_components=2).fit(X)

print(f"sklearn 의 분산 (= 고유값):  {np.round(pca.explained_variance_, 4)}")
print(f"sklearn 의 components (행이 v₁, v₂):")
print(np.round(pca.components_, 4))

print(f"\n수동 계산의 고유값:           {np.round(eigvals, 4)}")
print(f"수동 계산의 고유벡터 (열):")
print(np.round(eigvecs, 4))



### 3.3  주성분 방향을 시각화한다

자료의 산점도 위에 두 주성분 축을 길이 $\sqrt{\lambda}$ 로 그려, 분산이 가장 큰 방향을 확인한다.


In [ ]:
mean = X.mean(axis=0)

fig, ax = plt.subplots(figsize=(7, 6))
ax.scatter(X[:, 0], X[:, 1], color="#1B1F2A", alpha=0.4, s=20)

# 두 주성분 축. 화살표 길이 = sqrt(λ) * 2 (가시화용 배율)
scale = 2.5
for k in range(2):
    v = eigvecs[:, k]
    length = np.sqrt(eigvals[k]) * scale
    color = "#1E2761" if k == 0 else "#E0A11B"
    ax.annotate(
        "", xy=mean + v * length, xytext=mean,
        arrowprops=dict(arrowstyle="->", color=color, lw=2.5),
    )
    ax.text(*(mean + v * (length + 0.3)), f"PC{k+1}\nλ={eigvals[k]:.2f}",
            color=color, fontsize=12, fontweight="bold")

ax.set_aspect("equal")
ax.set_xlabel("x1")
ax.set_ylabel("x2")
ax.set_title("PCA — 분산을 최대로 보존하는 방향이 곧 고유벡터")
ax.grid(alpha=0.3)
plt.tight_layout()
plt.show()



### 3.4  사영(projection) 과 분산 비율

원 자료를 두 주성분 축에 사영하면 새 좌표 $Z = X V$ 가 얻어진다. 사영된 자료의 분산은 정확히 고유값과 같다.


In [ ]:
Z = X_centered @ eigvecs   # n × 2

print(f"사영 후 분산:           {Z.var(axis=0, ddof=1).round(4)}")
print(f"고유값과 비교:           {eigvals.round(4)}")
print(f"전체 분산 = trace(Σ):    {np.trace(Sigma).round(4)}")
print(f"분산 비율:               {(eigvals / eigvals.sum()).round(4)}")
print(f"누적 분산 비율:          {np.cumsum(eigvals / eigvals.sum()).round(4)}")



## 4.  다차원 사례 — iris 자료

scikit-learn 의 iris 자료에는 4 개 변수가 있다. PCA 로 2 차원에 사영해 보면 96% 의 분산이 보존된다.


In [ ]:
from sklearn.datasets import load_iris
from sklearn.preprocessing import StandardScaler

iris = load_iris()
X4 = iris.data
y4 = iris.target

# 단위가 다르므로 표준화 필수
Xs = StandardScaler().fit_transform(X4)

pca4 = PCA(n_components=4).fit(Xs)
print(f"고유값:        {pca4.explained_variance_.round(4)}")
print(f"분산 비율:      {pca4.explained_variance_ratio_.round(4)}")
print(f"누적 비율:      {np.cumsum(pca4.explained_variance_ratio_).round(4)}")


In [ ]:
# 2차원으로 사영
Z2 = PCA(n_components=2).fit_transform(Xs)

fig, axes = plt.subplots(1, 2, figsize=(11, 4.5))

# Scree plot
axes[0].plot(np.arange(1, 5), pca4.explained_variance_, "o-",
             color="#1E2761", linewidth=2.5, markersize=10)
axes[0].axhline(1.0, color="#E0A11B", linestyle="--", label="Kaiser λ=1")
axes[0].set_xlabel("주성분 번호")
axes[0].set_ylabel("고유값 λ")
axes[0].set_title("Scree plot")
axes[0].legend()
axes[0].grid(alpha=0.3)

# 2차원 사영
colors = {"setosa": "#1E2761", "versicolor": "#E0A11B", "virginica": "#6D2E46"}
for k, name in enumerate(iris.target_names):
    sub = Z2[y4 == k]
    axes[1].scatter(sub[:, 0], sub[:, 1], color=colors[name],
                    label=name, alpha=0.75, s=45, edgecolor="white")
axes[1].set_xlabel(f"PC1 ({pca4.explained_variance_ratio_[0]*100:.1f}%)")
axes[1].set_ylabel(f"PC2 ({pca4.explained_variance_ratio_[1]*100:.1f}%)")
axes[1].set_title("iris — 처음 두 주성분으로 사영")
axes[1].legend()
axes[1].grid(alpha=0.3)

plt.tight_layout()
plt.show()



## 5.  PCA 의 네 가지 사용 시점

1. **차원 축소** — 변수가 너무 많을 때 정보 손실을 최소화하며 줄인다.
2. **시각화** — 고차원 자료를 2~3 차원으로 사영해 군집·이상치를 본다.
3. **다중공선성 해결** — 회귀의 설명변수들이 강한 상관을 가질 때 주성분을 회귀변수로 쓴다 (주성분 회귀, PCR).
4. **잡음 제거** — 분산이 작은 후순위 주성분은 잡음일 가능성이 크므로 잘라낸다.

위 네 가지가 아닌 자리에서 PCA 를 쓰면 \"써 보긴 했지만 왜 썼는지 모르는\" 분석이 된다.



## 6.  연습문제

1. 다음 두 명제를 식으로 증명하라.
   - (a) $\boldsymbol{v}^{\mathsf T}\Sigma\boldsymbol{v} = \lambda$ when $\Sigma\boldsymbol{v} = \lambda\boldsymbol{v}$ and $\boldsymbol{v}^{\mathsf T}\boldsymbol{v} = 1$.
   - (b) 두 주성분 $\boldsymbol{v}_i, \boldsymbol{v}_j$ ($i \neq j$) 는 직교한다 (즉 $\boldsymbol{v}_i^{\mathsf T}\boldsymbol{v}_j = 0$).
2. 표준화하지 않고 PCA 를 적용했을 때 \"단위가 큰 변수가 결과를 지배\" 하는 현상을 본 노트북의 자료에 직접 단위를 곱해 재현하라.
3. iris 자료에서 \"setosa\" 한 품종만 선택하여 PCA 를 적용해보고, 처음 두 주성분의 설명력이 어떻게 달라지는지 적어라.
4. **OLS 와의 비교**: 같은 자료 $\{(x_i, y_i)\}$ 에 대해 (i) OLS 회귀의 기울기 $\hat\beta_1$, (ii) PCA 의 제 1 주성분 방향의 기울기를 각각 계산하고 비교하라. 어느 쪽이 더 큰가? 왜 그런가?
